# M2 Round 3B — VAR Baseline with BIC Lag Selection

**Issue:** #29  
**Owner:** Mitchel  
**Reviewer:** Lerneir  
**Branch:** `artifact/m2-round3b-baseline-var-bic`

## Objective

This notebook implements Round 3 Path B of the baseline-model comparison.

The goal is to evaluate a Vector Autoregression (VAR) model using lag order selected by the Bayesian Information Criterion (BIC), and compare its forecast accuracy against the shared Random Walk (naïve) benchmark.

To preserve a controlled AIC-vs-BIC comparison with Issue #28, the dataset, feature set, transformations, forecast horizons, evaluation procedure, benchmark, and metrics should remain the same. The intended experimental difference is the lag-order selection criterion.

**Fix (per #29 review):** an earlier version of this notebook included `usdcad` in the feature set while `#28`'s script didn't, which confounded the AIC-vs-BIC comparison with an extra input variable. The feature set was matched down to `#28`'s 5-variable set, and a Diebold-Mariano significance test (matching `#28`'s, including the `dm_report()` fix from `#43`/`#44`) was added so both paths were reported at the same rigor.

**Aug 19 correction:** that "fix" matched the wrong direction. Proposal §3.1's research question names USD/CAD explicitly as a required transmission variable, so dropping it to match `#28` moved both baselines away from the proposal's committed methodology instead of toward it. `#28` has now been corrected to include `usdcad` too (see `src/EDA_VAR_AIC _lag _order.py`), so this notebook restores it here rather than the other way around. See `M2_CHECKLIST.md`'s Aug 19 entry.

## Common Round 3 feature set

- `yield_spread_10y_2y`
- `overnight_rate`
- `us_treasury_10y`
- `fed_funds_rate`
- `cpi_yoy`
- `usdcad`

## Round 2 findings carried into this notebook

The merged Round 2 EDA found that the candidate series are non-stationary

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.api import VAR

import sys

# Bootstrap guess, just precise enough to import project_paths -- immediately
# replaced below by the authoritative, marker-based find_project_root(), so a
# wrong guess here (e.g. this notebook run from a different working directory)
# doesn't silently propagate into every downstream path.
_bootstrap_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_bootstrap_root / "src"))
from project_paths import find_project_root
PROJECT_ROOT = find_project_root()
PROCESSED = PROJECT_ROOT / "data" / "processed"

# Round 3 feature set — matches #28's script exactly, per proposal §3.1/§5.3's
# committed predictor set (usdcad restored Aug 19; see M2_CHECKLIST.md)
FEATURES = [
    "yield_spread_10y_2y",
    "overnight_rate",
    "us_treasury_10y",
    "fed_funds_rate",
    "cpi_yoy",
    "usdcad",
]

TARGET = "yield_spread_10y_2y"

HORIZONS = [1, 5, 20]

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED)

Project root: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5
Processed data: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/data/processed


In [2]:
# Load processed datasets
boc = pd.read_csv(
    PROCESSED / "bank_of_canada_data.csv",
    parse_dates=["date"]
)

fred = pd.read_csv(
    PROCESSED / "fred_rates.csv",
    parse_dates=["date"]
)

cpi = pd.read_csv(
    PROCESSED / "statcan_cpi.csv",
    parse_dates=["reference_month", "release_date"]
)

print("BoC:", boc.shape)
print("FRED:", fred.shape)
print("CPI:", cpi.shape)

BoC: (4552, 12)
FRED: (4563, 3)
CPI: (210, 3)


In [3]:
# Merge BoC + FRED on date
daily = (
    boc.merge(fred, on="date", how="inner")
       .sort_values("date")
       .reset_index(drop=True)
)

# Compute CPI YoY at monthly frequency BEFORE expanding to daily
cpi_monthly = cpi.sort_values("reference_month").copy()
cpi_monthly["cpi_yoy"] = (
    cpi_monthly["cpi_all_items"].pct_change(12) * 100
)

# Align CPI by release date to avoid look-ahead bias
cpi_daily = (
    cpi_monthly[
        ["release_date", "cpi_all_items", "cpi_yoy"]
    ]
    .rename(columns={"release_date": "date"})
    .sort_values("date")
)

daily = (
    daily.merge(cpi_daily, on="date", how="left")
         .sort_values("date")
         .reset_index(drop=True)
)

# Once CPI is publicly available, carry the latest released value forward
daily["cpi_all_items"] = daily["cpi_all_items"].ffill()
daily["cpi_yoy"] = daily["cpi_yoy"].ffill()

print("Daily merged shape:", daily.shape)
print("Date range:", daily["date"].min(), "->", daily["date"].max())

daily[
    [
        "date",
        "yield_spread_10y_2y",
        "overnight_rate",
        "usdcad",
        "us_treasury_10y",
        "fed_funds_rate",
        "cpi_yoy",
    ]
].tail()

Daily merged shape: (4552, 16)
Date range: 2009-01-02 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y,overnight_rate,usdcad,us_treasury_10y,fed_funds_rate,cpi_yoy
4547,2026-06-24,0.63,2.25,1.4234,4.41,3.63,3.225806
4548,2026-06-25,0.64,2.25,1.4204,4.40,3.63,3.225806
4549,2026-06-26,0.64,2.25,1.4186,4.38,3.63,3.225806
4550,2026-06-29,0.64,2.25,1.4206,4.38,3.63,3.225806
4551,2026-06-30,0.64,2.25,1.4210,4.44,3.63,3.225806


In [4]:
# Keep only the agreed Round 3 feature set
# and remove incomplete level observations BEFORE differencing
model_levels = (
    daily[["date"] + FEATURES]
    .dropna(subset=FEATURES)
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

# First-difference each series once
model_diff = model_levels.copy()

for col in FEATURES:
    model_diff[col] = model_diff[col].diff()

# Differencing creates one NaN in the first row only
model_diff = (
    model_diff
    .dropna(subset=FEATURES)
    .reset_index(drop=True)
)

print("Level sample shape:", model_levels.shape)
print("Modeling sample shape:", model_diff.shape)
print(
    "Modeling date range:",
    model_diff["date"].min(),
    "->",
    model_diff["date"].max()
)

model_diff.head()

Level sample shape: (4010, 7)
Modeling sample shape: (4009, 7)
Modeling date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y,overnight_rate,us_treasury_10y,fed_funds_rate,cpi_yoy,usdcad
0,2010-02-19,-0.01,0.0,-0.01,0.01,0.0,-0.0032
1,2010-02-22,0.01,0.0,0.02,-0.01,0.0,0.0008
2,2010-02-23,-0.02,0.0,-0.11,0.00,0.0,0.0089
3,2010-02-24,0.01,0.0,0.01,-0.01,0.0,0.0033
4,2010-02-25,0.01,0.0,-0.06,0.01,0.0,0.0125


In [5]:
# Separate dates from stationary model inputs
model_dates = model_diff["date"].copy()

var_data = (
    model_diff[FEATURES]
    .copy()
)

print("VAR input shape:", var_data.shape)
print("Columns:", list(var_data.columns))

VAR input shape: (4009, 6)
Columns: ['yield_spread_10y_2y', 'overnight_rate', 'us_treasury_10y', 'fed_funds_rate', 'cpi_yoy', 'usdcad']


In [6]:
# Select VAR lag order using BIC
MAX_LAGS = 15

lag_selection = VAR(var_data).select_order(maxlags=MAX_LAGS)

print(lag_selection.summary())
print("\nSelected lag by BIC:", lag_selection.bic)

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -41.42     -41.41*   1.030e-18     -41.41*
1       -41.42      -41.35   1.030e-18      -41.39
2       -41.42      -41.29   1.032e-18      -41.37
3       -41.42      -41.24   1.031e-18      -41.35
4       -41.41      -41.17   1.039e-18      -41.32
5       -41.43      -41.13   1.021e-18      -41.32
6       -41.42      -41.07   1.032e-18      -41.29
7       -41.41      -41.00   1.041e-18      -41.26
8       -41.41      -40.95   1.036e-18      -41.25
9       -41.41      -40.89   1.033e-18      -41.23
10     -41.55*      -40.97  9.022e-19*      -41.34
11      -41.55      -40.91   9.048e-19      -41.32
12      -41.54      -40.85   9.099e-19      -41.30
13      -41.53      -40.78   9.193e-19      -41.27
14      -41.53      -40.72   9.215e-19      -41.24
15      -41.52      -40.66   9.263e-19      -41.22
-------------------------------

In [7]:
# Shared evaluation configuration
MIN_TRAIN = 500
STEP = 5

BIC_LAG = int(lag_selection.bic)

print("BIC lag:", BIC_LAG)
print("Minimum training observations:", MIN_TRAIN)
print("Evaluation step:", STEP)
print("Horizons:", HORIZONS)

BIC lag: 0
Minimum training observations: 500
Evaluation step: 5
Horizons: [1, 5, 20]


In [8]:
# Align target levels exactly to the final modeling dates
aligned_levels = (
    model_levels[
        model_levels["date"].isin(model_diff["date"])
    ][["date", TARGET]]
    .sort_values("date")
    .reset_index(drop=True)
)

# Safety check: dates must match row-by-row
assert len(aligned_levels) == len(model_diff)
assert aligned_levels["date"].equals(model_diff["date"])

print("Aligned observations:", len(aligned_levels))
print(
    "Aligned date range:",
    aligned_levels["date"].min(),
    "->",
    aligned_levels["date"].max()
)

Aligned observations: 4009
Aligned date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


In [9]:
results = []

target_idx = FEATURES.index(TARGET)

for origin in range(MIN_TRAIN, len(model_diff) - max(HORIZONS), STEP):
    train_diff = model_diff.iloc[:origin][FEATURES].copy()

    # Level corresponding exactly to the forecast origin
    last_level = aligned_levels.loc[origin - 1, TARGET]

    if BIC_LAG == 0:
        # VAR(0): constant expected change estimated from training data
        target_mean_change = train_diff[TARGET].mean()

    else:
        var_model = VAR(train_diff)
        var_fit = var_model.fit(BIC_LAG)

    for h in HORIZONS:
        # h observations ahead from the origin
        actual_level = aligned_levels.loc[origin + h - 1, TARGET]

        # Random Walk
        naive_forecast = last_level

        # VAR-BIC
        if BIC_LAG == 0:
            var_forecast = last_level + h * target_mean_change
        else:
            forecast_diff = var_fit.forecast(
                train_diff.values[-BIC_LAG:],
                steps=h
            )

            cumulative_target_change = forecast_diff[:, target_idx].sum()
            var_forecast = last_level + cumulative_target_change

        results.append({
            "origin_date": aligned_levels.loc[origin - 1, "date"],
            "horizon": h,
            "actual": actual_level,
            "var_bic": var_forecast,
            "naive": naive_forecast,
        })

results_df = pd.DataFrame(results)

print("Forecast rows:", len(results_df))
print(results_df.groupby("horizon").size())

results_df.head()

Forecast rows: 2094
horizon
1     698
5     698
20    698
dtype: int64


,origin_date,horizon,actual,var_bic,naive
0,2012-03-02,1,0.87,0.857480,0.86
1,2012-03-02,5,0.84,0.847400,0.86
2,2012-03-02,20,0.91,0.809600,0.86
3,2012-03-09,1,0.82,0.837465,0.84
4,2012-03-09,5,0.96,0.827327,0.84


In [10]:
metrics = []

for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]

    var_rmse = np.sqrt(
        mean_squared_error(subset["actual"], subset["var_bic"])
    )
    var_mae = mean_absolute_error(
        subset["actual"], subset["var_bic"]
    )

    naive_rmse = np.sqrt(
        mean_squared_error(subset["actual"], subset["naive"])
    )
    naive_mae = mean_absolute_error(
        subset["actual"], subset["naive"]
    )

    metrics.append({
        "horizon": h,
        "var_bic_rmse": var_rmse,
        "naive_rmse": naive_rmse,
        "var_bic_mae": var_mae,
        "naive_mae": naive_mae,
        "rmse_improvement_pct": (
            (naive_rmse - var_rmse) / naive_rmse * 100
        ),
        "mae_improvement_pct": (
            (naive_mae - var_mae) / naive_mae * 100
        ),
    })

metrics_df = pd.DataFrame(metrics)

metrics_df.round(6)

,horizon,var_bic_rmse,naive_rmse,var_bic_mae,naive_mae,rmse_improvement_pct,mae_improvement_pct
0,1,0.030260,0.030238,0.022414,0.022321,-0.074525,-0.418329
1,5,0.063648,0.063416,0.048551,0.048438,-0.366051,-0.232014
2,20,0.132449,0.130718,0.100227,0.098825,-1.324500,-1.418246


In [11]:
# Diebold-Mariano test — is the RMSE/MAE gap vs. naive statistically significant, or noise?
# Mirrors #28's dm_report() (src/EDA_VAR_AIC _lag _order.py), including the #43/#44 fix:
# report RMSE- and MAE-loss verdicts separately, plus an overall verdict requiring both to agree.
from dieboldmariano import dm_test

ALPHA = 0.05

dm_rows = []
for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]
    actual = subset["actual"].to_numpy()
    var_pred = subset["var_bic"].to_numpy()
    naive_pred = subset["naive"].to_numpy()

    dm_rmse, p_rmse = dm_test(
        actual, var_pred, naive_pred,
        loss=lambda u, v: (u - v) ** 2,
        h=h, harvey_correction=True, variance_estimator="bartlett",
    )
    dm_mae, p_mae = dm_test(
        actual, var_pred, naive_pred,
        loss=lambda u, v: abs(u - v),
        h=h, harvey_correction=True, variance_estimator="bartlett",
    )

    var_better_rmse = bool(p_rmse < ALPHA and dm_rmse < 0)
    naive_better_rmse = bool(p_rmse < ALPHA and dm_rmse > 0)
    var_better_mae = bool(p_mae < ALPHA and dm_mae < 0)
    naive_better_mae = bool(p_mae < ALPHA and dm_mae > 0)

    dm_rows.append({
        "horizon_days": h,
        "dm_stat_squared_loss": round(dm_rmse, 3),
        "dm_p_value_squared_loss": round(p_rmse, 4),
        "dm_stat_absolute_loss": round(dm_mae, 3),
        "dm_p_value_absolute_loss": round(p_mae, 4),
        "var_significantly_better_rmse": var_better_rmse,
        "naive_significantly_better_rmse": naive_better_rmse,
        "var_significantly_better_mae": var_better_mae,
        "naive_significantly_better_mae": naive_better_mae,
        "var_significantly_better": bool(var_better_rmse and var_better_mae),
        "naive_significantly_better": bool(naive_better_rmse and naive_better_mae),
    })

dm_results_df = pd.DataFrame(dm_rows)

for _, row in dm_results_df.iterrows():
    h = int(row["horizon_days"])
    if row["var_significantly_better"]:
        verdict = "VAR-BIC significantly better than naive (both RMSE and MAE agree)"
    elif row["naive_significantly_better"]:
        verdict = "naive significantly better than VAR-BIC (both RMSE and MAE agree)"
    elif (row["var_significantly_better_rmse"] and row["naive_significantly_better_mae"]) or \
         (row["naive_significantly_better_rmse"] and row["var_significantly_better_mae"]):
        verdict = "mixed result: RMSE and MAE are both significant but identify different winning models"
    elif row["var_significantly_better_rmse"] or row["naive_significantly_better_rmse"]:
        winner = "VAR-BIC" if row["var_significantly_better_rmse"] else "naive"
        verdict = (f"{winner} significantly better on RMSE only "
                   f"(p_rmse={row['dm_p_value_squared_loss']:.4f}) -- MAE test not significant")
    elif row["var_significantly_better_mae"] or row["naive_significantly_better_mae"]:
        winner = "VAR-BIC" if row["var_significantly_better_mae"] else "naive"
        verdict = (f"{winner} significantly better on MAE only "
                   f"(p_mae={row['dm_p_value_absolute_loss']:.4f}) -- RMSE test not significant")
    else:
        verdict = "no significant difference on either RMSE or MAE"
    print(f"  h={h}: {verdict}")

dm_results_df

  h=1: naive significantly better on MAE only (p_mae=0.0124) -- RMSE test not significant
  h=5: no significant difference on either RMSE or MAE
  h=20: no significant difference on either RMSE or MAE


,horizon_days,dm_stat_squared_loss,dm_p_value_squared_loss,dm_stat_absolute_loss,dm_p_value_absolute_loss,var_significantly_better_rmse,naive_significantly_better_rmse,var_significantly_better_mae,naive_significantly_better_mae,var_significantly_better,naive_significantly_better
0,1,0.664,0.5070,2.507,0.0124,False,False,False,True,False,False
1,5,1.307,0.1917,0.606,0.5446,False,False,False,False,False,False
2,20,1.171,0.2421,1.023,0.3068,False,False,False,False,False,False


In [12]:
# Save Round 3B results
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

metrics_output = OUTPUT_DIR / "r3_pathb_var_bic_vs_naive.csv"
forecasts_output = OUTPUT_DIR / "r3_pathb_var_bic_forecasts.csv"
dm_output = OUTPUT_DIR / "r3_pathb_diebold_mariano.csv"

metrics_df.to_csv(metrics_output, index=False)
results_df.to_csv(forecasts_output, index=False)
dm_results_df.to_csv(dm_output, index=False)

print("Saved metrics:", metrics_output)
print("Saved forecasts:", forecasts_output)
print("Saved Diebold-Mariano results:", dm_output)

Saved metrics: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/outputs/r3_pathb_var_bic_vs_naive.csv
Saved forecasts: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/outputs/r3_pathb_var_bic_forecasts.csv
Saved Diebold-Mariano results: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/outputs/r3_pathb_diebold_mariano.csv


## Round 3B Conclusion — VAR with BIC Lag Selection

Using the Round 3 feature set aligned with `#28` — `yield_spread_10y_2y`, `overnight_rate`, `us_treasury_10y`, `fed_funds_rate`, `cpi_yoy`, and `usdcad` — all six variables were modeled in first differences based on the Round 2 stationarity findings. `usdcad` was restored on Aug 19 after a prior "fix" (per #29 review) had matched this notebook *down* to `#28`'s 5-variable set instead of the other way around; proposal §3.1's research question names USD/CAD explicitly as a required transmission variable, so `#28`'s script has now been corrected to include it too (see `M2_CHECKLIST.md`).

To ensure direct comparability with Path A, incomplete level observations were removed before differencing. This produced **4,010 complete level observations and 4,009 differenced modeling observations** -- identical counts to the 5-variable run, since `usdcad`'s date coverage doesn't further restrict the sample beyond what CPI's monthly-release alignment already limits.

For the BIC-based VAR specification, lag-order selection over a maximum of 15 lags again selected:

- **BIC lag order: 0**

This indicates that, under the BIC penalty, adding autoregressive lags did not provide enough additional explanatory value to justify the increase in model complexity. At lag 0, the specification reduces to a constant expected daily change estimated from the training window and does not use lagged values from the other variables.

The BIC-selected specification was evaluated using an expanding-window approach with forecasts generated every 5 observations after an initial training sample of 500 observations. Performance was assessed at **1-day, 5-day, and 20-day horizons** using RMSE and MAE, with a Random Walk forecast used as the shared naïve benchmark.

### Forecast performance

| Horizon | VAR-BIC RMSE | Random Walk RMSE | VAR-BIC MAE | Random Walk MAE |
|---|---:|---:|---:|---:|
| 1 day | 0.030260 | 0.030238 | 0.022414 | 0.022321 |
| 5 days | 0.063648 | 0.063416 | 0.048551 | 0.048438 |
| 20 days | 0.132449 | 0.130718 | 0.100227 | 0.098825 |

The BIC specification did **not outperform the Random Walk benchmark at any forecast horizon**. Relative to the naïve benchmark, the VAR-BIC model produced slightly higher errors:

- **1-day horizon:** RMSE +0.07%, MAE +0.42%
- **5-day horizon:** RMSE +0.37%, MAE +0.23%
- **20-day horizon:** RMSE +1.32%, MAE +1.42%

These numbers are numerically identical to the 5-variable run: with `BIC_LAG == 0`, the forecast is `last_level + h * train_diff[TARGET].mean()` -- a constant-drift model computed purely from the target's own training-window history. It never reads `usdcad` (or any other regressor) at all, so adding `usdcad` back to the feature set couldn't change these particular results. It does matter for the *lag-order search itself* (all 6 series enter the AIC/BIC criterion table above) and for direct comparability with `#28`'s now-6-variable feature set.

### Diebold-Mariano significance

To assess whether the numerical differences between VAR-BIC and the Random Walk benchmark were statistically meaningful, the same Diebold-Mariano methodology used in `#28` was applied separately to squared-error and absolute-error losses.

| Horizon | DM stat (squared loss) | p (squared loss) | DM stat (absolute loss) | p (absolute loss) | Verdict |
|---|---:|---:|---:|---:|---|
| 1 day | 0.664 | 0.5070 | 2.507 | 0.0124 | naive significantly better on MAE only |
| 5 days | 1.307 | 0.1917 | 0.606 | 0.5446 | no significant difference |
| 20 days | 1.171 | 0.2421 | 1.023 | 0.3068 | no significant difference |

At the **1-day horizon**, the Random Walk is statistically significantly more accurate under absolute-error loss (`p = 0.0124`), while the squared-error test is not significant. At the 5-day and 20-day horizons, neither loss function shows a statistically significant difference between VAR-BIC and the Random Walk.

Overall, BIC strongly favored model parsimony and selected a zero-lag specification. The resulting forecasts remained very close to the Random Walk benchmark, but the Random Walk retained slightly lower RMSE and MAE at all three horizons. The Diebold-Mariano test provides additional evidence at the 1-day horizon that the naïve benchmark performs better under MAE loss.

These results provide the Path B baseline for the team comparison between **AIC- and BIC-based VAR lag selection**, now on the proposal-correct, mutually comparable 6-variable feature set shared with `#28` and `#49`.